# CTIG v1.4 — Xem từng bước của pipeline

**Settings (panel phải):** Accelerator = GPU T4 x2 (hoặc T4), Internet = On.

Mỗi cell dưới đây hiện đầu ra của MỘT bước. Badge góc trên bảng cho biết cell vừa lấy kết quả từ **bộ nhớ**, **đĩa** hay **chạy mới** (chạy lại cell không đổi gì thì không tốn API/model). Đổi `PROMPT_ID` hoặc config rồi chạy lại từ cell prompt.

| Bước | Hiện gì |
|---|---|
| 1 | prompt tách thành keywords bề mặt / keywords mới, thực thể ứng viên |
| 2 | search text + ảnh: cột **từ keywords** và cột **từ prompt gốc**, top-K, điểm CLIP |
| 2b | bằng chứng đưa vào spec: KB viết tay so với thuộc tính rút từ web (có câu gốc) |
| 3 | CulturalSpec và GenSpec (prompt / negative thật sự đưa vào bộ sinh) |
| 4 | nhiều model sinh ảnh cùng GenSpec, grid hàng = model |
| 5 | bảng điểm CLIP identity, BLIP-2 ITM, CLIP sim |
| 5b | (tuỳ chọn) vòng review agent cũ |

In [ ]:
REPO_URL = "https://github.com/OxyzGiaHuy/CTIG.git"
REPO     = "/kaggle/working/CTIG"
CONFIG   = "configs/kaggle_walkthrough_t4x2.yaml"   # 1 GPU: configs/kaggle_walkthrough.yaml

import os, subprocess
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
%cd $REPO
!git pull -q
!git log --oneline -1
# Nếu ctig đã import ở phiên trước (kernel chưa restart) thì gỡ khỏi bộ nhớ để nạp bản mới vừa pull
import sys
for _m in [k for k in sys.modules if k.startswith("ctig")]:
    del sys.modules[_m]
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# torchao 0.10 có sẵn trên Kaggle làm peft/transformers import lỗi (cần >=0.16); không dùng nên gỡ.
!pip uninstall -q -y torchao 2>/dev/null; cd $REPO && pip install -q -r requirements.txt && pip install -q -e . --no-deps
import transformers, diffusers; print("transformers", transformers.__version__, "| diffusers", diffusers.__version__)

## Key (không bắt buộc)

| Nguồn | Cần gì | Lấy ở đâu |
|---|---|---|
| DuckDuckGo (mặc định) | không | — |
| Civitai LoRA áo dài | `CIVITAI_TOKEN` | civitai.com → ảnh đại diện → Account settings → API Keys → Add. Kaggle: Add-ons → Secrets |
| Serper (Google web + images) | `SERPER_API_KEY` | serper.dev → Sign up → Dashboard → API Key. Rồi `--set retrieval.web_api=serper` |
| Model gated (SD3) | `HF_TOKEN` | huggingface.co → Settings → Access Tokens, và bấm chấp nhận điều khoản trên trang model |

Cell dưới đọc secret nếu có, không có thì bỏ qua.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for k in ("CIVITAI_TOKEN", "SERPER_API_KEY", "HF_TOKEN", "ANTHROPIC_API_KEY"):
        try:
            v = _s.get_secret(k)
            if v:
                os.environ[k] = v; print("đã đọc", k)
        except Exception:
            pass
except Exception:
    print("không có kaggle_secrets (chạy ngoài Kaggle)")
print("CIVITAI_TOKEN:", "có" if os.getenv("CIVITAI_TOKEN") else "không -> hàng sdxl_aodai sẽ bị bỏ qua")

In [ ]:
# Khôi phục cache từ Dataset đã Add Input. Kaggle TỰ GIẢI NÉN zip khi upload, nên trong dataset
# thường là thư mục runs/_cache/... chứ không còn file zip; cell này xử cả hai trường hợp.
import glob, os, shutil, zipfile

DATASET = None   # ví dụ "/kaggle/input/datasets/<user>/runs-cache"; None = tự tìm trong /kaggle/input
dst = "/kaggle/working"

def merge_tree(src, dst):
    n = 0
    for root, _, files in os.walk(src):
        rel = os.path.relpath(root, src)
        out = os.path.join(dst, rel) if rel != "." else dst
        os.makedirs(out, exist_ok=True)
        for f in files:
            if not os.path.exists(os.path.join(out, f)):
                shutil.copy2(os.path.join(root, f), os.path.join(out, f)); n += 1
    return n

cands = [DATASET] if DATASET else [os.path.dirname(x) for x in glob.glob("/kaggle/input/**/runs_cache.zip", recursive=True)] \
        + [os.path.dirname(os.path.dirname(x)) for x in glob.glob("/kaggle/input/**/_cache", recursive=True)]
done = False
for ds in dict.fromkeys(c for c in cands if c):
    zips = glob.glob(f"{ds}/**/runs_cache.zip", recursive=True)
    if zips:
        for z in zips:
            print("giải nén", z); zipfile.ZipFile(z).extractall(dst)
        done = True
    elif os.path.isdir(f"{ds}/runs"):
        print("copy", merge_tree(f"{ds}/runs", f"{dst}/runs"), "file từ", ds); done = True
    elif os.path.isdir(f"{ds}/_cache"):
        print("copy", merge_tree(f"{ds}/_cache", f"{dst}/runs/_cache"), "file từ", ds); done = True
if not done:
    print("không có cache để khôi phục (lần đầu chạy, hoặc chưa Add Input dataset)")
cache = f"{dst}/runs/_cache"
print("cache hiện có:", {d: len(os.listdir(f"{cache}/{d}")) for d in sorted(os.listdir(cache))} if os.path.isdir(cache) else "chưa có")

## Chọn prompt

In [ ]:
PROMPT_ID = "p001"          # xem data/prompts_vi.jsonl; hoặc đặt TEXT
TEXT = None                 # ví dụ: "Một cụ ông đội khăn xếp thổi đàn bầu trước sân đình"
RUN_REVIEW = False          # True = chạy thêm vòng review agent ở cuối

from ctig.config import Config
from ctig.pipeline import load_prompts
from ctig.schema import Prompt
from ctig.session import Session
from ctig import viz

cfg = Config.load(CONFIG)
prompt = Prompt("adhoc", TEXT, TEXT) if TEXT else next(p for p in load_prompts(cfg.prompts_path) if p.id == PROMPT_ID)
s = Session(cfg, prompt, run_dir="/kaggle/working/runs/walkthrough")
report = viz.Report(f"CTIG walkthrough · {prompt.id}")
report.show(viz.prompt_card(prompt))

## Bước 1 · Prompt → keywords
Bảng trên: thực thể được nêu tên thẳng. Bảng dưới: thứ agent suy ra thêm, có lý do. Cảnh báo đỏ nếu agent đổ cả KB vào (đã cap ở stage 1).

In [ ]:
a, src = s.analysis()
report.show(viz.keywords_table(a, s.kb, cfg.max_spec_entities, source=src))

## Bước 2 · Search: keywords vs prompt gốc
Cùng một hệ search (DuckDuckGo tiếng Việt + tiếng Anh, Commons), chỉ khác câu hỏi. Ảnh xếp theo CLIP sim với prompt; cột keywords có thêm P(đúng thực thể).
Đây là chỗ trả lời câu hỏi: bước phân tích có giúp search ra đúng thứ hơn không?

In [ ]:
cmp, src = s.compare()
report.show(viz.query_comparison(cmp, cfg.search_viz.k_text, cfg.search_viz.k_images, source=src))

## Bước 2b · Bằng chứng đưa vào spec
Cột trái: KB viết tay. Cột phải: thuộc tính VLM rút từ văn bản web, mỗi dòng kèm câu gốc. Ảnh viền xanh = được chọn làm tham chiếu IP-Adapter.

In [ ]:
search, src = s.retrieve()
report.show(viz.evidence_table(search, s.kb, source=src))

## Bước 2c · Summary agent
Tóm tắt tư liệu vừa truy hồi của mỗi thực thể thành brief thị giác (facts EN/VI, khác gì với thứ dễ nhầm, một câu 'vẽ thế nào'). Chỉ được dùng thông tin trong văn bản; facts VI được kiểm mờ có câu gốc. Brief dùng cho Rank agent và (khi `agents.enrich_prompt=true`) nối vào prompt kiểu Culture-TRIP.

In [ ]:
briefs, src = s.brief()
report.show(viz.brief_card(briefs, s.spec()[0], source=src))

## Bước 3 · CulturalSpec → GenSpec
Cột EN là thứ thật sự đi vào prompt SDXL (lấy từ KB viết tay, không dịch bằng VLM).

In [ ]:
spec, src = s.spec()
report.show(viz.spec_card(spec, source=src))
gen, src = s.genspec()
report.show(viz.genspec_card(gen, source=src))

In [ ]:
# 1xT4: giải phóng Qwen trước khi nạp model sinh ảnh. 2xT4 có thể bỏ qua cell này.
if cfg.multigen.device == cfg.llm.device:
    s.free_vlm()
viz.show(viz.vram_html())

## Bước 4 · Nhiều model cùng một GenSpec (v1.3: best-of-N, hires, PickScore)
Model nạp tuần tự (nhỏ trước), mỗi hàng hiện ngay khi xong. Dưới tên model có **số token** của prompt và **ghi chú** (scheduler, LoRA scale, compel, hires, IP-Adapter). Hàng `*_aodai` chỉ chạy khi spec có áo dài và có `CIVITAI_TOKEN`; hàng `sdxl_refplus` chỉ chạy khi có ảnh tham chiếu đạt CLIP.

Muốn sweep LoRA scale: `s.multigen(["sdxl_aodai@0.6", "sdxl_aodai@0.8", "sdxl_aodai@1.0"])` (mỗi khoá một hàng, cùng seed).

Hàng `sdxl_refplus` lấy ảnh tham chiếu đã qua **Filter agent** (bỏ ảnh nhóm khi prompt một người, bỏ ảnh không có must_have); bảng lọc hiện ngay bên dưới grid.

In [ ]:
from ctig.models.registry import get
from ctig.schema import MultiGenResult
mg = cfg.multigen
print(f"n_candidates={mg.n_candidates} · scheduler={mg.scheduler} · compel={mg.long_prompt} · hires={'x'+str(mg.hires.scale) if mg.hires.enabled else 'tắt'} · PickScore={mg.aesthetic.enabled} · ref_images={mg.ref_images}")
for k in cfg.models:
    m = get(k); print(f"{k:16} {m.repo:45} {m.width}px {m.steps} bước g{m.guidance:g} ~{m.est_vram_gb} GB  {m.notes[:70]}")

res, src = s.multigen(cfg.models, on_model_done=lambda r: viz.show(viz.model_grid(MultiGenResult(prompt.id, [r]), spec)))
report.show(viz.model_grid(res, spec, source=src))
viz.show(viz.vram_html())

# Bước 3b: ảnh tham chiếu sau Filter agent (chỉ khi có hàng IP-Adapter và agents.ref_filter bật)
if cfg.agents.enabled and cfg.agents.ref_filter and "ref_filter" in s.steps:
    report.show(viz.filter_table(s.steps["ref_filter"].value, title="Bước 3b · Filter agent trên ảnh tham chiếu", source=s.steps["ref_filter"].source))

## Bước 5 · Bảng điểm
CLIP identity: P(ảnh giống mô tả thực thể Việt) so với các confusable. ITM: BLIP-2, họ model khác reviewer. sim: cosine CLIP với prompt tiếng Anh. Ba số để xếp thứ tự, kết luận vẫn phải nhìn grid.

In [ ]:
report.show(viz.score_table(res, source=src))

## Bước 4c–4d · Agent loop review (v1.4): Filter → Rank → một vòng sửa
Đúng ba ô trong sơ đồ gốc. **Filter agent**: VLM chỉ *mô tả* từng ảnh top-k (số người, trang phục, vật, nền), rồi một agent văn bản so mô tả với must_have/must_not và phải trích được cụm trong mô tả; ảnh có must_not hoặc sai số người bị bỏ. **Rank agent** xếp phần còn lại từ mô tả + brief, đối chiếu với xếp hạng theo metric (top-1 trùng không, Spearman). Ứng viên đầu còn lỗi thì lập kế hoạch sửa bằng luật (thiếu gì thêm vào prompt, thấy must_not gì thêm vào negative) và sinh lại **một** lần trên model tốt nhất; chỉ đổi ảnh cuối khi ảnh mới sạch hơn thật. Trên 1×T4 cell này nạp lại Qwen (~1 phút).

In [ ]:
cr, src = s.candidate_review()
report.show(viz.candidate_review_html(cr, source=src))

## Bước 5b · Vòng review agent (tuỳ chọn)
Tắt mặc định vì VLM 3B trả lời checklist thiên lệch (xem research/research-log.md). Đặt `RUN_REVIEW = True` ở cell prompt để chạy.

In [ ]:
if RUN_REVIEW:
    outcome, src = s.review()
    report.show(viz.review_summary(outcome, spec, source=src))
else:
    print("bỏ qua (RUN_REVIEW=False)")

## Xuất
`walkthrough.html` là toàn bộ các bảng phía trên trong một file (ảnh đã nhúng). `runs_cache.zip` để lần sau khôi phục cache.

In [ ]:
out = report.save(s.out_dir / "walkthrough.html")
# Zip CẢ ảnh đã sinh (runs/walkthrough) lẫn cache, vì Kaggle xoá /kaggle/working giữa hai phiên; thiếu ảnh cũ thì bước 4 sinh lại toàn bộ.
!cd /kaggle/working && rm -f runs_cache.zip && zip -qr runs_cache.zip runs/_cache runs/walkthrough -x '*/grid_hires.png' && ls -lh runs_cache.zip
from IPython.display import FileLink, display
display(FileLink(str(out)))
display(FileLink(str(s.out_dir / "grid.png")))
display(FileLink("/kaggle/working/runs_cache.zip"))
print("llm cache:", __import__('ctig.llm.cache', fromlist=['current']).current().stats())

# Báo cáo tiến độ gửi người hướng dẫn: HTML tự chứa, ảnh độ phân giải gốc, bảng/biểu đồ vector, kèm grid_hires.png cho slide.
from ctig.progress_report import build
build(str(s.run_dir), "/kaggle/working/progress_report.html",   # gom MỌI prompt đã chạy trong runs/walkthrough
      title="CTIG - Cultural T2I (Việt Nam): báo cáo tiến độ", cfg=cfg)
display(FileLink("/kaggle/working/progress_report.html")); display(FileLink(str(s.out_dir / "grid_hires.png")))